# Dialogue State Tracking

DST keeps the slot-value dict in sync so the booking works.

## Problem definition

In a task-oriented dialogue system, the user's goal is encoded as a set of slot-value pairs: `{cuisine: italian, area: north, price: moderate}`. Every user turn can add, change, or remove a slot. The system must read the whole conversation and output the current state correctly.

Why it matters.
1. Compliance-sensitive domains require deterministic slot values, not free-form generation.
2. Tool-use agents still need slot resolution before calling APIs.
3. Multi-turn correction is harder than it looks: "Acutally no, make it Thursday"

## Basic Concept

DST: dialog history --> slot-value state

### Task structure

A schema defines domains(restaurant, hotel, taxi) and their slots (cuisione, are, price, people). Each slot can be
1. empty.
2. filled with a value from a closed set.
3. free-form value.

### Two DST formulations

1. Classification.   
    For each (slot, candidate_value) pair, predict yes or no.
2. Generation.
    Given the dialogue, generate slot values as free text.

### Mertric

Joint Goal Accuracy (JGA)

the fraction of turns where **every slot is correct**.

## Architecture

1. Rule-based (slot regex + keyword). Strong baseline for narrow domains.
2. TripPy/BERT-DST.  Copy-based generation with BERT encoding.
3. LDST. Instruction-**tuned** LLM with domain-slot prompting.
4. Ontology-free.  Skip the schemas, generate slot names and values directly.
5. Prompt + structured output.  LLM with Pydantic schema + constrained decoding.

## Failure modes

1. Co-reference across turns.
2. Over-write vs append.
3. Implicit confirmations.
4. Correction.
5. Coreference to previous system utterance.

# Build your Own

## Rule-based slot extractor

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

CUISINE_SYNONYMS = {
    "italian": ["italian", "pasta", "pizza", "italy"],
    "chinese": ["chinese", "chow mein", "noodles"]
}

def extract_cuisine(utterance):
    for canonical, synonyms in CUISINE_SYNONYMS.items():
        if any(word in utterance.lower() for word in synonyms):
            return canonical
    return "unknown"

with SectionPrinter("Rule-based slot extractor"):
    print(extract_cuisine("I'd like to eat at a pizza place"))
    print(extract_cuisine("I'm in the mood for some Chinese food"))
    print(extract_cuisine("I'm looking for a restaurant in the area"))




=================Rule-based slot extractor==================
italian
chinese
unknown


## State update loop

In [ ]:
"""
def update_state(state, utterance):
    new_state = dict(state)
    for slot, extractor in SLOT_EXTRACTORS.items():
        value = extractor(utterance)
        if value is not None:
            new_state[slot] = value
    for slot in NEGATION_CLEARS:
        if is_negated(utterance, slot):
            new_state[slot] = None
    return new_state
"""


## LLM-driven DST with strctured output

In [2]:
import sys
from pathlib import Path
from typing import Literal, Optional

from pydantic import BaseModel

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter, load_project_env

load_project_env()


class RestaurantState(BaseModel):
    cuisine: Optional[Literal["italian", "chinese", "unknown"]] = None
    area: Optional[Literal["north", "south", "east", "west", "center"]] = None
    price: Optional[Literal["cheap", "moderate", "expensive"]] = None
    people: Optional[int] = None
    day: Optional[str] = None


SLOT_FIELDS = list(RestaurantState.model_fields.keys())


def missing_slots(state: RestaurantState) -> list[str]:
    return [slot for slot in SLOT_FIELDS if getattr(state, slot) is None]


def is_complete(state: RestaurantState) -> bool:
    return not missing_slots(state)


from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model

agent = create_agent(
    init_chat_model(
        "deepseek:deepseek-chat",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    response_format=ToolStrategy(
        RestaurantState,
        tool_message_content="Extracted dialog state.",
    ),
    system_prompt=(
        "You are a dialog state tracker for restaurant booking. "
        "Given the full conversation, output the cumulative slot values. "
        "Keep previously filled slots unless the user changes or cancels them."
    ),
)

def merge_state(prev: RestaurantState, update: RestaurantState) -> RestaurantState:
    data = prev.model_dump()
    for slot in SLOT_FIELDS:
        value = getattr(update, slot)
        if value is not None:
            data[slot] = value
    return RestaurantState(**data)


USER_TURNS = [
    "I'd like to eat at a pizza place in the north area.",
    "Something moderately priced, please.",
    "There will be 4 of us.",
    "How about this Friday?",
]

messages: list[dict[str, str]] = []
state = RestaurantState()

with SectionPrinter("LLM-driven DST with structured output"):
    for turn, user_text in enumerate(USER_TURNS, start=1):
        if is_complete(state):
            break

        messages.append({"role": "user", "content": user_text})
        response = agent.invoke({"messages": messages})
        messages = response["messages"]
        if response.get("structured_response"):
            state = merge_state(state, response["structured_response"])

        print(f"\n--- Turn {turn} ---")
        print(f"User: {user_text}")
        print(f"State: {state.model_dump()}")
        missing = missing_slots(state)
        print(f"Missing: {missing if missing else 'none (complete)'}")

        if is_complete(state):
            print("\nAll slots filled. Booking can proceed.")
            break

    print(f"\nFinal state: {state}")


===========LLM-driven DST with structured output============
Extracted dialog state.
cuisine='italian' area='north' price=None people=None day=None
